# Data Protection with MS Presidio

## Introduction

Microsoft Presidio is an open-source toolkit for detecting and de-identifying PII (Personally Identifiable Information) like names, emails, phone numbers, credit cards, and so on in free text. It works in two stages:
- Analyzer: finds PII. It runs a `spaCy` NLP model to catch context-dependent entities (like a person's name) and combines that with regex patterns and checksums for structured entities (like an IBAN or credit card).
- Anonymizer: acts on what was found. It replaces, masks, hashes, redacts, or encrypts each detected span.

`spaCy` is the NLP engine underneath. Presidio relies on a `spaCy` language model to understand sentence structure and recognize named entities. That's why the two are installed together.

The bottom-up path: 1. detect one entity, 2. inspect the raw result, 3. anonymize it. Then layer on more entities, custom recognizers, and different anonymization strategies.

In [2]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

text = "My name is Peter and my email is peter123@example.com"
results = AnalyzerEngine().analyze(text=text, language="en")
print(results)  # should list a PERSON and an EMAIL_ADDRESS span

[type: EMAIL_ADDRESS, start: 33, end: 53, score: 1.0, type: PERSON, start: 11, end: 16, score: 0.85, type: URL, start: 42, end: 53, score: 0.5]


## How multi-language works in Presidio

Presidio doesn't understand languages on its own. It relies on `spaCy` models, and each language has its own model you must install separately. English used `en_core_web_lg`; Dutch needs the Dutch model:

```
python -m spacy download nl_core_news_lg
```

Then there are two places the language has to be declared — this trips people up:

The NLP engine must be told which model to load for which language. The default `AnalyzerEngine()` only loads English, so for Dutch you build the engine explicitly with an `NlpEngineProvider` configuration.
The `.analyze()` call must be passed `language="nl"` so Presidio knows which loaded model to use and which recognizers apply.

Miss either one and you'll get an error or zero detections.

In [4]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine

# 1. Tell the NLP engine which spaCy model to use for Dutch
configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "nl", "model_name": "nl_core_news_lg"}],
}
nlp_engine = NlpEngineProvider(nlp_configuration=configuration).create_engine()

# 2. Build the analyzer with that engine + declare it supports Dutch
analyzer = AnalyzerEngine(
    nlp_engine=nlp_engine,
    supported_languages=["nl"],
)

# 3. Analyze — note language="nl"
text = """Hallo, mijn naam is Peter de Vries. Ik werk bij het bedrijf NovaTech Solutions. \\
Mijn twee collega’s heten Lisa Jansen en Mark Smit. \\
Ons bedrijf is gevestigd aan de Parkstraat 24, 2312 AB Leiden. \\
Het telefoonnummer van mijn collega Lisa is 06-12345678."""

results = analyzer.analyze(text=text, language="nl")

for r in results:
    print(r, "->", text[r.start:r.end])

# 4. Anonymize
anonymizer = AnonymizerEngine()
anonymized = anonymizer.anonymize(text=text, analyzer_results=results)
print("\n", anonymized.text)

type: PERSON, start: 20, end: 34, score: 0.85 -> Peter de Vries
type: ORGANIZATION, start: 60, end: 78, score: 0.85 -> NovaTech Solutions
type: PERSON, start: 108, end: 119, score: 0.85 -> Lisa Jansen
type: PERSON, start: 123, end: 132, score: 0.85 -> Mark Smit
type: DATE_TIME, start: 183, end: 187, score: 0.85 -> 2312
type: ORGANIZATION, start: 188, end: 197, score: 0.85 -> AB Leiden
type: PERSON, start: 237, end: 241, score: 0.85 -> Lisa
type: DATE_TIME, start: 245, end: 256, score: 0.85 -> 06-12345678
type: PHONE_NUMBER, start: 245, end: 256, score: 0.4 -> 06-12345678

 Hallo, mijn naam is <PERSON>. Ik werk bij het bedrijf <ORGANIZATION>. \
Mijn twee collega’s heten <PERSON> en <PERSON>. \
Ons bedrijf is gevestigd aan de Parkstraat 24, <DATE_TIME> <ORGANIZATION>. \
Het telefoonnummer van mijn collega <PERSON> is <DATE_TIME>.


## Why isn't the address fully masked?

Running the default analyzer on the Dutch text leaves the address line leaking:
`Parkstraat 24, 2312 AB Leiden` — `Leiden` becomes `<LOCATION>`, but the house
number `24` and postcode `2312 AB` are missed, or grabbed as `DATE_TIME`.

**Two reasons:**
- **No built-in `ADDRESS` entity.** Presidio covers PERSON, LOCATION, EMAIL,
  PHONE, IBAN… but not street + house number. Nothing claims `24`.
- **Loose numbers drift to `DATE_TIME`.** The date recognizer is greedy — `24`
  and `2312` look date-like, so it grabs them with a *low score*: a false positive.

**Fixes:** raise the score threshold (drop noise) and/or add custom
`PatternRecognizer`s (add missing PII). Real pipelines use both.

In [5]:
from presidio_analyzer import Pattern, PatternRecognizer

def show(results, text):
    """See every detection with its confidence score — the key tuning habit."""
    for r in sorted(results, key=lambda r: r.start):
        print(f"{r.entity_type:18} score={r.score:.2f}  ->  {text[r.start:r.end]!r}")

# --- Baseline: reproduce the problem (note the DATE_TIME false positives) ---
show(analyzer.analyze(text=text, language="nl"), text)

# --- Fix 1: raise the threshold to drop weak detections ---
# analyzer.analyze(text=text, language="nl", score_threshold=0.5)

# --- Fix 2: teach Presidio the missing PII via custom recognizers ---
nl_postcode = PatternRecognizer(
    supported_entity="NL_POSTCODE", supported_language="nl",
    patterns=[Pattern(name="nl_postcode", regex=r"\b\d{4}\s?[A-Z]{2}\b", score=0.9)],
)
nl_street = PatternRecognizer(
    supported_entity="NL_STREET_ADDRESS", supported_language="nl",
    patterns=[Pattern(name="nl_street",
        regex=r"\b[A-Z][a-z]+(?:straat|laan|weg|plein|kade|gracht|dijk|hof|steeg)\s+\d+\b",
        score=0.85)],
)
analyzer.registry.add_recognizer(nl_postcode)
analyzer.registry.add_recognizer(nl_street)

# --- Result: custom recognizers + threshold together ---
results = analyzer.analyze(text=text, language="nl", score_threshold=0.4)
show(results, text)
print("\n--- anonymized ---")
print(anonymizer.anonymize(text=text, analyzer_results=results).text)

PERSON             score=0.85  ->  'Peter de Vries'
ORGANIZATION       score=0.85  ->  'NovaTech Solutions'
PERSON             score=0.85  ->  'Lisa Jansen'
PERSON             score=0.85  ->  'Mark Smit'
DATE_TIME          score=0.85  ->  '2312'
ORGANIZATION       score=0.85  ->  'AB Leiden'
PERSON             score=0.85  ->  'Lisa'
DATE_TIME          score=0.85  ->  '06-12345678'
PHONE_NUMBER       score=0.40  ->  '06-12345678'
PERSON             score=0.85  ->  'Peter de Vries'
ORGANIZATION       score=0.85  ->  'NovaTech Solutions'
PERSON             score=0.85  ->  'Lisa Jansen'
PERSON             score=0.85  ->  'Mark Smit'
NL_STREET_ADDRESS  score=0.85  ->  'Parkstraat 24'
NL_POSTCODE        score=0.90  ->  '2312 AB'
DATE_TIME          score=0.85  ->  '2312'
ORGANIZATION       score=0.85  ->  'AB Leiden'
PERSON             score=0.85  ->  'Lisa'
DATE_TIME          score=0.85  ->  '06-12345678'
PHONE_NUMBER       score=0.40  ->  '06-12345678'

--- anonymized ---
Hallo, mijn naam i

## Understanding the `score`

Every detection Presidio returns comes with a **score between 0.0 and 1.0**.
It's a **confidence level** — *"how sure am I this really is PII?"*

- `1.0` = completely certain
- `0.5` = a maybe
- `0.0` = no confidence

Think of it like a smoke detector's sensitivity, not a measure of "how private" something is.

### Where the score comes from

Different recognizers earn their confidence differently:

- **Checksum / validated patterns → high score (~0.95–1.0).**
  A credit card or IBAN isn't just digits — it passes a math check. If it passes,
  Presidio is almost certain. Little room for coincidence.

- **Regex patterns → the score *you* assign.**
  When you write a `PatternRecognizer`, *you* pick the number. A strict pattern
  that can't match by accident deserves a high score; a loose one deserves a low score.
    - Dutch postcode `\d{4}\s?[A-Z]{2}` → very specific → `score=0.9`
    - "any 9 digits" → could be many things → `score=0.4`

- **NLP / spaCy model (names, places) → the model decides (~0.85).**
  For a PERSON, Presidio trusts the language model's judgement. Fixed-ish, not tunable per word.

### Why it matters: the threshold

`score_threshold` is a cutoff — **keep detections at or above it, drop the rest.**

Example — suppose the analyzer returns:

| text        | entity      | score |
|-------------|-------------|-------|
| Peter       | PERSON      | 0.85  |
| 2312 AB     | NL_POSTCODE | 0.90  |
| 24          | DATE_TIME   | 0.35  |

- `score_threshold=0.5` → keeps Peter + postcode, **drops the `24` false positive** ✅
- `score_threshold=0.9` → keeps only the postcode — now you're **over-filtering** and losing Peter ⚠️

### How to choose your values (rules of thumb)

- **The more specific the pattern, the higher the score.** Can it match by accident? Low. Almost never? High.
- **Start around 0.85** for a solid custom pattern, then adjust after testing.
- **Set the threshold by your priority:**
    - Catch *everything*, tolerate some noise (privacy-first) → **low threshold** (e.g. 0.3)
    - Only mask when confident, tolerate some misses → **high threshold** (e.g. 0.6+)
- **There is no single "correct" number** — you tune it against *your* data by inspecting scores.

> Rule to remember: **the recognizer proposes a score, the threshold disposes of it.**

## When a *non-sensitive* field becomes sensitive: indirect identifiers

Imagine a situation that you are asked to mask a course name. 
A course name is usually **not** personal data — "Introduction to Databases"
tells you nothing about a person. So why mask it?

**The context makes it sensitive.** The educational team wants to analyse *why*
students get frustrated, across many feedback texts. But each course is taught by
one known teacher — so **naming the course indirectly names the teacher**. An
analyst reading the feedback could single out that teacher, fairly or not.

This is a **quasi-identifier** (indirect identifier): a value that isn't
identifying by itself, but points to a person *in this context*. Postcodes,
job titles, and course names all behave this way.

**The lesson:** sensitivity isn't a property of the data alone — it depends on
**who reads it and what they can link it to**. Presidio can't judge this for you;
*you* decide what counts as PII for your use case, and add a recognizer for it.

Here we mask the course name so the team can study the **reasons** for frustration
(here: *lack of clear planning*) without exposing the teacher behind the course.

## How the recognizer solves this

A **recognizer** is Presidio's way of teaching the system *what to look for*.
When we add one for the course name, we give Presidio a new entity it now knows
how to find and mask. Two techniques do this:

- **Pattern matching (regex):** describe the *shape* of the value, not the value
  itself. Good when items share a form but you don't know each one in advance —
  e.g. a course *code* `INF-2312` via `\b[A-Z]{2,4}-\d{3,4}\b`. One rule covers
  many unseen values.

- **Deny list:** give Presidio the *exact values* to mask. Good when you already
  have the list — e.g. course *names* like "Introduction to Databases". No pattern
  needed; it matches those strings directly.

**Rule of thumb:** For known *shape* use **pattern**; For known *values* use **deny list**. Pattern
matching generalises to new cases; a deny list is exact but must be kept up to date.

In [6]:
from presidio_analyzer import PatternRecognizer

feedback = """Ik ben erg teleurgesteld over de cursus Introduction to Databases.
Er was totaal geen duidelijke planning: opdrachten werden op het laatste moment
aangekondigd en deadlines veranderden steeds. Hierdoor wist ik nooit waar ik aan
toe was. De inhoud was interessant, maar door de chaotische organisatie raakte ik
gefrustreerd en gedemotiveerd."""

# Treat the course name as a quasi-identifier for THIS analysis
course_recognizer = PatternRecognizer(
    supported_entity="COURSE_NAME",
    supported_language="nl",
    deny_list=["Introduction to Databases", "Object Oriented Programming"],
)
analyzer.registry.add_recognizer(course_recognizer)

results = analyzer.analyze(text=feedback, language="nl")
show(results, feedback)   # reuse the show() helper from earlier
print("\n--- anonymized (safe to analyse for reasons) ---")
print(anonymizer.anonymize(text=feedback, analyzer_results=results).text)

COURSE_NAME        score=1.00  ->  'Introduction to Databases'

--- anonymized (safe to analyse for reasons) ---
Ik ben erg teleurgesteld over de cursus <COURSE_NAME>.
Er was totaal geen duidelijke planning: opdrachten werden op het laatste moment
aangekondigd en deadlines veranderden steeds. Hierdoor wist ik nooit waar ik aan
toe was. De inhoud was interessant, maar door de chaotische organisatie raakte ik
gefrustreerd en gedemotiveerd.


## Three important features of Presidio

### 1. Anonymization operators — not just `<REPLACE>`
The anonymizer can *act* on detected PII in several ways, per entity type:

- **replace** → `<PERSON>` (default)
- **mask** → keep part, hide the rest, e.g. `06-****5678`
- **redact** → delete it entirely
- **hash** → a consistent scrambled token (same input → same hash)
- **encrypt** → reversible with a key

```python
from presidio_anonymizer.entities import OperatorConfig
anonymizer.anonymize(
    text=text, analyzer_results=results,
    operators={"PHONE_NUMBER": OperatorConfig("mask",
        {"masking_char": "*", "chars_to_mask": 6, "from_end": False})},
)
```
**Why it matters:** *hash* lets you still count/link records (same person → same token)
without revealing them; *encrypt* lets you recover the original later. You choose the
trade-off between privacy and utility.

### 2. Context-aware recognition
A recognizer can **raise its confidence** when trigger words sit nearby. `123456`
alone is weak; `patiëntnummer 123456` should score higher. `PatternRecognizer`
accepts a `context=[...]` list of words that boost the score when found close by.
This reduces false positives *and* false negatives using the surrounding sentence.

### 3. Beyond plain text — images & structured data
Presidio isn't only for free text. Companion packages extend it:

- **presidio-image-redactor** → find and black-out PII in *images/scanned docs* (via OCR)
- **presidio-structured** → apply the same detection to *tables / DataFrames / JSON*

Same detection engine, different data shapes — so one mental model covers documents,
spreadsheets, and scans.

## Summary

Microsoft Presidio is a customizable, open-source framework for detecting and de-identifying personal data (PII) in text. It works in two clear stages: the **Analyzer** *finds* PII — combining a spaCy **NLP engine** (for context-dependent entities like names and places) with a **registry of recognizers** (regex patterns, checksums, and your own custom pattern or deny-list recognizers), and returning each finding with a confidence **score**; the **Anonymizer** then *acts* on those findings, applying an operator such as replace, mask, redact, hash, or encrypt. Because every recognizer is modular, you can extend Presidio to your own domain — Dutch postcodes, course codes, or a quasi-identifier like a course name — without disturbing the built-ins. The same engine also reaches beyond plain text to images and structured tables through companion packages, and encryption can be reversed with the Deanonymizer. The result is a pipeline you tune to your data: decide *what* counts as sensitive, *how confident* you must be, and *how* to hide it.

### Architectural view

![Presidio architecture](pix/presidio_architecture.png)